### 1. Configuração Inicial e Criação do Schema Bronze
Nesta etapa, importamos as bibliotecas necessárias do PySpark e Python para lidar com manipulação de tempo e requisições web. Em seguida, apontamos o ambiente para o catálogo dedicado do projeto (`cinedata_analytics`) e garantimos a criação do schema `bronze`. Esta camada é responsável por armazenar o dado em seu estado bruto, exatamente como extraído da origem.

In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from datetime import datetime, timedelta

# 1. Criar o banco de dados (database) da camada Bronze
spark.sql("USE CATALOG cinedata_analytics")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

DataFrame[]

### 2. Ingestão dos Arquivos Estáticos (CSVs)
Este bloco realiza a leitura automatizada dos 5 arquivos disponibilizados pela área de negócios e armazenados no Volume do Unity Catalog. O script itera sobre um dicionário de mapeamento e, para cada arquivo:
* Lê os dados mantendo a estrutura e os tipos originais (`inferSchema=True`).
* Cria uma coluna de auditoria (`ingestion_datetime`) com o timestamp exato da carga para garantir rastreabilidade.
* Salva os registros como tabelas gerenciadas no formato **Delta** utilizando o modo `append`, o que preserva o histórico de cargas sem sobrescrever dados anteriores.

In [0]:
# Substitui o caminho abaixo pelo caminho real do teu volume (ex: /Volumes/workspace/default/nome_do_volume/)
volume_path = "/Volumes/cinedata_analytics/bronze/cinedata_inputs/"

# Mapeamento exigido: Arquivo Original -> Nome da Tabela Bronze
mapeamento_arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

for arquivo, nome_tabela in mapeamento_arquivos.items():
    caminho_completo = f"{volume_path}{arquivo}"
    
    # Ler os dados sem qualquer alteração estrutural ou de conteúdo
    df_csv = spark.read.csv(caminho_completo, header=True, inferSchema=True, multiLine=True, escape='"')
    
    # Adicionar a coluna ingestion_datetime contendo o timestamp exato da inserção
    df_bronze = df_csv.withColumn("ingestion_datetime", current_timestamp())
    
    # Gravar a tabela em formato Delta, utilizando o modo Append no database bronze
    df_bronze.write.format("delta").mode("append").saveAsTable(f"bronze.{nome_tabela}")
    
    print(f"Tabela bronze.{nome_tabela} carregada com sucesso a partir de {arquivo}.")

Tabela bronze.tb_movies_info carregada com sucesso a partir de movies_info_TMDB_IMDB.csv.
Tabela bronze.tb_movies_financials carregada com sucesso a partir de movies_financials_IMDB_TMDB.csv.
Tabela bronze.tb_movies_metrics carregada com sucesso a partir de movies_metrics_IMDB_TMDB.csv.
Tabela bronze.tb_credits_and_tags carregada com sucesso a partir de credits_and_tags_IMDB_TMDB.csv.
Tabela bronze.tb_movies_reviews carregada com sucesso a partir de movies_reviews.csv.


### 3. Extração via API (Banco Central do Brasil)
Para atender à necessidade de conversão de moedas (USD para BRL), este bloco extrai a cotação do dólar (PTAX) diretamente da API oficial do Banco Central. O processo é parametrizado:
* Calcula dinamicamente a janela de busca para os últimos 7 dias.
* Utiliza parâmetros (`dbutils.widgets`) no topo do notebook, permitindo a alteração das datas de execução sob demanda.
* Realiza a requisição HTTP e converte a resposta JSON em um DataFrame PySpark.
* Adiciona o timestamp de ingestão e grava os dados na tabela `bronze.tb_cotacao_dolar`.

In [0]:
# Calcular datas sugeridas (hoje e 7 dias atrás)
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

# Formatar para MM-DD-AAAA conforme exigido
data_fim_padrao = hoje.strftime("%m-%d-%Y")
data_inicio_padrao = sete_dias_atras.strftime("%m-%d-%Y")

# Criar os parâmetros (widgets) no topo do notebook
dbutils.widgets.text("data_inicio", data_inicio_padrao, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_padrao, "Data Fim (MM-DD-AAAA)")

# Capturar os valores inseridos nos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Montar o Endpoint da API com as datas formatadas
url_bcb = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# Extrair os dados da API
resposta = requests.get(url_bcb)
dados_api = resposta.json()
valores_cotacao = dados_api.get("value", [])

if valores_cotacao:
    # Criar um DataFrame Spark a partir do JSON recebido
    df_cotacao = spark.createDataFrame(valores_cotacao)
    
    # Adicionar timestamp de ingestão
    df_cotacao_bronze = df_cotacao.withColumn("ingestion_datetime", current_timestamp())
    
    # Gravar em formato Delta, modo Append na tabela bronze.tb_cotacao_dolar
    df_cotacao_bronze.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
    print("Tabela bronze.tb_cotacao_dolar carregada com sucesso via API!")
else:
    print(f"Nenhuma cotação encontrada para o período entre {data_inicio} e {data_fim}.")

Tabela bronze.tb_cotacao_dolar carregada com sucesso via API!
